# LangGraph Persistence Explained (with Real-World Examples)

## What is Persistence in LangGraph?

**Persistence** is the ability of a LangGraph application to **save its current state** so it can continue later from where it stopped.

Think of it like a video game.

- **Without Save Feature:** If the game crashes, you start from Level 1.
- **With Save Feature:** You continue from your last checkpoint.

LangGraph works the same way.

```text
Run Graph
    │
    ▼
 Node A
    │
    ▼
 Node B
    │
    ▼
 Save State (Persistence)
    │
    ▼
 Node C
```

If the application crashes after **Node B**, LangGraph can resume from **Node C** instead of starting over.

---

# Why Do We Need Persistence?

Imagine you're building a **Flight Booking AI Assistant**.

### Conversation

```
User: I want to book a flight.

Bot: Which city are you departing from?

User: Lahore

Bot: What's your destination?

User: Dubai
```

## Without Persistence

The assistant forgets previous messages.

```
User: Dubai

Bot: Hello! How can I help you today?
```

Everything is lost.

---

## With Persistence

The assistant remembers:

```python
origin = "Lahore"
destination = "Dubai"
```

So it continues naturally:

```
Bot: Great! When do you want to travel?
```

---

# What Does LangGraph Save?

LangGraph saves the **State**.

Example:

```python
state = {
    "messages": [...],
    "origin": "Lahore",
    "destination": "Dubai",
    "current_step": "asking_travel_date"
}
```

The next time the graph runs, it loads this state and continues from there.

---

# What is a Checkpointer?

A **Checkpointer** is the component responsible for **saving and loading the graph state**.

Think of it as the **Save button** in a video game.

```text
Graph
   │
   ▼
Execute Node
   │
   ▼
Checkpointer Saves State
   │
   ▼
Database / Memory / File
```

Every time a node finishes executing, the updated state is saved.

---

## Example Without a Checkpointer

```text
Run

Node 1
 ↓
Node 2
 ↓
Application Crashes

Restart

Node 1
 ↓
Node 2
```

Everything starts over.

---

## Example With a Checkpointer

```text
Run

Node 1
 ↓
Node 2
 ↓
Save State

Application Crashes

Restart

Continue from Node 3
```

Only the remaining work is executed.

---

# Types of Checkpointers

## 1. MemorySaver

Stores state in RAM.

```python
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
```

### Pros

- Very fast
- Great for testing
- Easy to use

### Cons

- Data is lost when the Python program stops.

---

## 2. SQLite Checkpointer

Stores state inside a SQLite database.

```text
state.db
```

### Pros

- Data survives application restarts
- Good for local applications

---

## 3. PostgreSQL Checkpointer

Stores state in PostgreSQL.

```text
Graph
   │
   ▼
PostgreSQL Database
   │
   ▼
Persistent State
```

### Best for

- Production systems
- Enterprise applications
- Multi-user chatbots
- AI assistants

---

# What are Threads?

A **Thread** represents a unique conversation or workflow.

Each thread has its own saved state.

Think of WhatsApp.

```
Ali
Ahmed
Sara
```

Each person has a separate chat history.

Similarly, LangGraph stores a separate state for every **thread_id**.

---

## Example

### User 1

```
Thread ID = user_001

Conversation:
Book a flight

Saved State:
Origin = Lahore
Destination = Dubai
```

---

### User 2

```
Thread ID = user_002

Conversation:
Order Pizza

Saved State:
Pizza = Large
Drink = Coke
```

The two users never share memory.

---

## Using thread_id

```python
config = {
    "configurable": {
        "thread_id": "user_001"
    }
}
```

When LangGraph receives another request with the same thread ID, it automatically loads the previous state.

---

# What Happens If thread_id Changes?

### First Request

```python
thread_id = "100"
```

Conversation:

```
Hello
```

State is saved.

---

### Second Request

```python
thread_id = "100"
```

LangGraph loads the saved conversation.

---

### Different Thread

```python
thread_id = "101"
```

LangGraph creates a completely new conversation.

No previous memory is loaded.

---

# How Persistence Works

```text
User Request
      │
      ▼
Load State using thread_id
      │
      ▼
Execute Current Node
      │
      ▼
Update State
      │
      ▼
Checkpointer Saves State
      │
      ▼
Return Response
```

Next request:

```text
User Request
      │
      ▼
Same thread_id
      │
      ▼
Load Saved State
      │
      ▼
Continue Graph Execution
```

---

# Real-World Example 1: Food Ordering Bot

Without Persistence

```
User: I want pizza.

Bot: Which size?

User: Large

Bot: Hello! What would you like to order?
```

The bot forgot everything.

---

With Persistence

```
User: I want pizza.

(State Saved)

User: Large

(State Loaded)

Bot: Which toppings would you like?
```

---

# Real-World Example 2: Bank Loan Application

Customer fills:

```
Step 1
Name
```

Saved.

```
Step 2
Salary
```

Saved.

Internet disconnects.

Next day:

```
Continue from Salary Step
```

Instead of restarting the application.

---

# Real-World Example 3: AI Coding Assistant

User:

```
Build a Flask REST API.
```

Thirty minutes later:

```
Add JWT Authentication.
```

Persistence remembers:

- Previous conversation
- Generated code
- Project architecture
- Existing files

The assistant continues instead of asking everything again.

---

# Real-World Example 4: Human Approval Workflow

```text
Generate Contract
       │
       ▼
Pause
       │
       ▼
Manager Reviews Tomorrow
       │
       ▼
Approve
       │
       ▼
Resume Workflow
```

Persistence keeps the workflow alive while waiting for human approval.

---

# Benefits of Persistence

- Maintains conversation history.
- Resumes execution after crashes.
- Supports long-running workflows.
- Enables human approval processes.
- Prevents repeating expensive LLM calls.
- Makes debugging easier.
- Supports multiple users simultaneously.
- Allows applications to restart without losing progress.

---

# Relationship Between Persistence, Checkpointer, Thread, and State

```text
                     Persistence
                          │
                          │
        ┌─────────────────┴─────────────────┐
        │                                   │
        ▼                                   ▼
   Checkpointer                     Thread ID
        │                                   │
        ▼                                   ▼
 Saves & Loads State          Identifies Which State
        │                                   │
        └──────────────┬────────────────────┘
                       ▼
                    Graph State
                       │
                       ▼
      Messages, Variables, Current Step, Tool Results
```

---

# Complete Flow

```text
                  User Sends Message
                          │
                          ▼
                 Thread ID Received
                          │
                          ▼
           Checkpointer Loads Saved State
                          │
                          ▼
              LangGraph Executes Next Node
                          │
                          ▼
                State Gets Updated
                          │
                          ▼
          Checkpointer Saves Updated State
                          │
                          ▼
                 Response Sent to User
```

---

# Summary Table

| Concept | Definition | Real-World Analogy |
|----------|------------|--------------------|
| **Persistence** | Saves the graph's state so execution can continue later. | Video game save system |
| **State** | The current data of the workflow (messages, variables, progress). | Player's game progress |
| **Checkpointer** | Component that saves and loads the state. | Save/Load button in a game |
| **Thread ID** | Unique identifier for a conversation or workflow. | WhatsApp chat ID |
| **MemorySaver** | Stores state in RAM. | Sticky note on your desk |
| **SQLite Checkpointer** | Stores state in a local SQLite database. | Personal notebook |
| **PostgreSQL Checkpointer** | Stores state in a production database. | Company records database |

---

# Key Takeaways

- **Persistence** allows LangGraph to remember progress between executions.
- **State** contains all the information the graph needs to continue.
- **Checkpointer** is responsible for saving and restoring the state.
- **Thread ID** identifies which conversation or workflow the state belongs to.
- Together, these features enable reliable, stateful AI applications such as chatbots, booking assistants, customer support systems, coding assistants, and long-running business workflows.

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
from langchain_groq import ChatGroq

In [ ]:
load_dotenv()

llm = ChatGroq(model="llama-3.3-70b-versatile")

In [3]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [101]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [102]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [103]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [104]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza go to the doctor? Because it was feeling a little saucy!',
 'explanation': 'This joke plays on the double meaning of the word "saucy." In one sense, "saucy" can mean bold, impertinent, or sassy. But in the context of a pizza, "saucy" refers to the tomato sauce typically used as a base on pizzas. So when the pizza went to the doctor because it was feeling "saucy," it implies that the pizza was not feeling well due to too much sauce, rather than being bold or sassy. The humor comes from the unexpected twist on the word\'s meaning.'}

In [105]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor? Because it was feeling a little saucy!', 'explanation': 'This joke plays on the double meaning of the word "saucy." In one sense, "saucy" can mean bold, impertinent, or sassy. But in the context of a pizza, "saucy" refers to the tomato sauce typically used as a base on pizzas. So when the pizza went to the doctor because it was feeling "saucy," it implies that the pizza was not feeling well due to too much sauce, rather than being bold or sassy. The humor comes from the unexpected twist on the word\'s meaning.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-93a2-6a08-8002-395e36be0f5e'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}, 'thread_id': '1'}, created_at='2025-07-29T21:56:42.071296+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-7a2f-60ea-8001-4ac26c539f8d'}}, tasks=(), inte

In [106]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor? Because it was feeling a little saucy!', 'explanation': 'This joke plays on the double meaning of the word "saucy." In one sense, "saucy" can mean bold, impertinent, or sassy. But in the context of a pizza, "saucy" refers to the tomato sauce typically used as a base on pizzas. So when the pizza went to the doctor because it was feeling "saucy," it implies that the pizza was not feeling well due to too much sauce, rather than being bold or sassy. The humor comes from the unexpected twist on the word\'s meaning.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-93a2-6a08-8002-395e36be0f5e'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}, 'thread_id': '1'}, created_at='2025-07-29T21:56:42.071296+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-7a2f-60ea-8001-4ac26c539f8d'}}, tasks=(), int

In [107]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti sit down at the dinner table? \nBecause it was pasta-tively exhausted from all that boiling!',
 'explanation': 'This joke plays on the idea that pasta needs to be boiled in order to be cooked and ready to eat. The punchline, "pasta-tively exhausted," is a play on words between "positively exhausted" and "pasta," highlighting the fact that the spaghetti was tired from being boiled. The humor lies in the personification of the spaghetti as if it has feelings and actions, such as sitting down at the dinner table.'}

In [108]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor? Because it was feeling a little saucy!', 'explanation': 'This joke plays on the double meaning of the word "saucy." In one sense, "saucy" can mean bold, impertinent, or sassy. But in the context of a pizza, "saucy" refers to the tomato sauce typically used as a base on pizzas. So when the pizza went to the doctor because it was feeling "saucy," it implies that the pizza was not feeling well due to too much sauce, rather than being bold or sassy. The humor comes from the unexpected twist on the word\'s meaning.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-93a2-6a08-8002-395e36be0f5e'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}, 'thread_id': '1'}, created_at='2025-07-29T21:56:42.071296+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-7a2f-60ea-8001-4ac26c539f8d'}}, tasks=(), inte

In [109]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor? Because it was feeling a little saucy!', 'explanation': 'This joke plays on the double meaning of the word "saucy." In one sense, "saucy" can mean bold, impertinent, or sassy. But in the context of a pizza, "saucy" refers to the tomato sauce typically used as a base on pizzas. So when the pizza went to the doctor because it was feeling "saucy," it implies that the pizza was not feeling well due to too much sauce, rather than being bold or sassy. The humor comes from the unexpected twist on the word\'s meaning.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-93a2-6a08-8002-395e36be0f5e'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}, 'thread_id': '1'}, created_at='2025-07-29T21:56:42.071296+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-7a2f-60ea-8001-4ac26c539f8d'}}, tasks=(), int

### Time Travel

In [110]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5"}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}, 'thread_id': '1'}, created_at='2025-07-29T21:56:38.565188+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-7230-65a8-bfff-0a96c2fc4e11'}}, tasks=(PregelTask(id='dcd96e38-1f32-5ed6-9f44-fa2b22c193f0', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the pizza go to the doctor? Because it was feeling a little saucy!'}),), interrupts=())

In [111]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5"}})

{'topic': 'pizza',
 'joke': 'Why did the mushroom go to the pizza party? Because he was a fungi and everyone wanted a pizza him!',
 'explanation': 'This joke plays on the word "fun guy" (fungi) which sounds like "fungi," a type of mushroom. The play on words is that the mushroom went to the pizza party because he was a "fun guy" and people wanted to "pizza" (see) him. The joke is a pun that combines the idea of mushrooms being fungi with the concept of being a fun person at a party.'}

In [112]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the mushroom go to the pizza party? Because he was a fungi and everyone wanted a pizza him!', 'explanation': 'This joke plays on the word "fun guy" (fungi) which sounds like "fungi," a type of mushroom. The play on words is that the mushroom went to the pizza party because he was a "fun guy" and people wanted to "pizza" (see) him. The joke is a pun that combines the idea of mushrooms being fungi with the concept of being a fun person at a party.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc70-100a-6bff-8002-7d6c3d37b1f4'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}, 'thread_id': '1'}, created_at='2025-07-29T21:57:21.959833+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc70-064c-630b-8001-707d60a085ad'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the mushroom go to the 

#### Updating State

In [113]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f06cc72-ca16-6359-8001-7eea05e07dd2'}}

In [114]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc72-ca16-6359-8001-7eea05e07dd2'}}, metadata={'source': 'update', 'step': 1, 'parents': {}, 'thread_id': '1'}, created_at='2025-07-29T21:58:35.155132+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, tasks=(PregelTask(id='0f085bb0-c1e8-d9fd-fb15-c427126b7cd6', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the mushroom go to the pizza party? Because he was a fungi and everyone wanted a pizza him!', 'explanation': 'This joke plays on the word "fun guy" (fungi) which sounds like "fungi," a type of mushroom. The play on words is that the mushroom went to the pizza party because he was a "fun guy" and people 

In [115]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc72-ca16-6359-8001-7eea05e07dd2"}})

{'topic': 'samosa',
 'joke': 'Why did the samosa bring a ladder to the party? \nBecause it wanted to be the best snack in the room and rise to the occasion!',
 'explanation': 'This joke plays on the double meaning of the word "rise." In one sense, "rise" means to physically move upwards, which is why the samosa brought a ladder to the party. However, in another sense, "rise" can also mean to perform well or excel, as in rising to the occasion. So, the samosa brought a ladder to symbolize its desire to physically rise above the other snacks at the party and also to metaphorically rise to the occasion by being the best snack in the room.'}

In [116]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the samosa bring a ladder to the party? \nBecause it wanted to be the best snack in the room and rise to the occasion!', 'explanation': 'This joke plays on the double meaning of the word "rise." In one sense, "rise" means to physically move upwards, which is why the samosa brought a ladder to the party. However, in another sense, "rise" can also mean to perform well or excel, as in rising to the occasion. So, the samosa brought a ladder to symbolize its desire to physically rise above the other snacks at the party and also to metaphorically rise to the occasion by being the best snack in the room.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc75-4407-6195-8003-b08dcfd27511'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}, 'thread_id': '1'}, created_at='2025-07-29T21:59:41.628661+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoin

### Fault Tolerance

In [4]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [3]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [4]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [5]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)


In [ ]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)

In [ ]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))